# SAMap supplementary figure, panel a: mouse pseudotime on the joint UMAP

The source of the transfer: every mouse nucleus of the SAMap alignment on the joint UMAP, colored by its pseudotime
(`pt_harmonized_rank`, the pseudotime of record). Human nuclei sit underneath in light grey. Apoptotic decidual
nuclei carry no pseudotime and are drawn in a darker grey, not imputed.

Pseudotime is drawn on one fixed 0-1 viridis scale with values as stored, never re-ranked (as in Fig. 6 and the
methods). Every nucleus is drawn.

**Inputs**, written by rule `samap_figure_tables`: `joint.parquet` (species and joint UMAP of every nucleus),
`mouse_cells.csv` and `human_cells.csv` (pseudotime, the transfer of record, labels).
**Outputs**, in `OUT`: `sfig_samap_panel_a.png` (300 dpi), `sfig_samap_panel_a_600dpi.png`,
`sfig_samap_panel_a.svg` (text kept as text) and `sfig_samap_panel_a.provenance.json`.
`sfig_samap_compose.ipynb` places the panel in the figure and adds its letter.

**Running it.** Run all cells with this directory as the working directory: the shared modules next to the notebook
are imported from there. The cells above *Run* only define constants and functions. Paths are relative to the
repository root. Existing outputs are not overwritten unless `FORCE = True`.

The pipeline runs it with papermill (`workflow/rules/10c_samap_figures.smk`). By hand, from the repository root:

```bash
papermill notebooks/samap/sfig_samap_a_umap_mouse_pseudotime.ipynb \
    results/notebooks/samap/sfig_samap_panel_a.executed.ipynb \
    --cwd notebooks/samap -p FORCE True
```

Commit this notebook without outputs: the provenance record notes whether the notebook matches its commit.

In [ ]:
# Parameters. papermill -p NAME VALUE overrides them in a cell below this one.
# Paths are relative to the repository root; REPO = None finds the root with git from the working directory.
REPO = None
JOINT = "results/samap_figures/tables/joint.parquet"
MOUSE_CELLS = "results/samap_figures/tables/mouse_cells.csv"
HUMAN_CELLS = "results/samap_figures/tables/human_cells.csv"
OUT = "results/figures/samap_supp/panels"
FORCE = False  # overwrite existing outputs
NOTEBOOK = "notebooks/samap/sfig_samap_a_umap_mouse_pseudotime.ipynb"  # this notebook, for the provenance record

In [ ]:
import logging
import sys

import matplotlib.pyplot as plt
from IPython.display import Image, display

import sfig_samap_style as style
from sfig_samap_umap import DOT, OTHER_SPECIES, load_umap_cells, umap_panel

# This notebook's messages in the cell output; other libraries stay at WARNING.
logging.basicConfig(format="%(message)s", stream=sys.stdout, force=True)
LOG = logging.getLogger("sfig_samap_a")
LOG.setLevel(logging.INFO)

## Constants

In [ ]:
LETTER = "a"
TITLE = "Mouse pseudotime"

# Apoptotic decidual nuclei have no pseudotime; a mid grey keeps them apart from the light-grey human nuclei.
APOPTOTIC = "#9a9a9a"  # as Fig. 6, NO_VALUE_UMAP
KEY = [(OTHER_SPECIES, "human cells"), (APOPTOTIC, "mouse, apoptotic\n(no value)")]

## Figure

In [ ]:
def draw_mouse_source(ax, cells: dict) -> None:
    """Human nuclei underneath in light grey, then mouse nuclei: apoptotic in grey, the rest by pseudotime."""
    human, mouse = cells["human"], cells["mouse"]
    ax.scatter(human.umap1, human.umap2, s=DOT, color=OTHER_SPECIES, linewidths=0, rasterized=True)

    apoptotic = mouse[mouse.pt_rank.isna()]
    valued = mouse[mouse.pt_rank.notna()]
    ax.scatter(apoptotic.umap1, apoptotic.umap2, s=DOT, color=APOPTOTIC, linewidths=0, rasterized=True)
    colors = style.PSEUDOTIME_CMAP(style.PSEUDOTIME_NORM(valued.pt_rank.values))
    ax.scatter(valued.umap1, valued.umap2, s=DOT, c=colors, linewidths=0, rasterized=True)

## Run

In [ ]:
repo = style.find_repo(REPO)
notebook = repo / NOTEBOOK
if not notebook.is_file():
    raise FileNotFoundError(f"{notebook} not found; set NOTEBOOK to this notebook's path")

paths = style.panel_paths(repo / OUT, LETTER)
style.refuse_overwrite(paths.values(), FORCE)
paths["png"].parent.mkdir(parents=True, exist_ok=True)

In [ ]:
cells = load_umap_cells(repo / JOINT, repo / MOUSE_CELLS, repo / HUMAN_CELLS)
mouse = cells["mouse"]
drawn_counts = {
    "mouse_with_pseudotime": int(mouse.pt_rank.notna().sum()),
    "mouse_apoptotic": int(mouse.pt_rank.isna().sum()),
    "human": len(cells["human"]),
}
LOG.info("Nuclei drawn: %s", drawn_counts)

In [ ]:
font = style.choose_font()
style.apply_style(font)
LOG.info("Font: %s", font)

fig = umap_panel(LETTER, TITLE, cells, lambda ax: draw_mouse_source(ax, cells), key=KEY)

style.save_panel(fig, LETTER, paths)
plt.close(fig)
LOG.info("Wrote %s", [str(path) for name, path in paths.items() if name != "provenance"])

# The saved PNG rather than the live figure, so what shows here is the file itself.
display(Image(filename=str(paths["png"]), width=420))

In [ ]:
record = style.provenance(
    notebook,
    parameters={
        "REPO": str(repo),
        "JOINT": JOINT,
        "MOUSE_CELLS": MOUSE_CELLS,
        "HUMAN_CELLS": HUMAN_CELLS,
        "OUT": OUT,
        "FORCE": FORCE,
    },
    inputs={"joint": repo / JOINT, "mouse_cells": repo / MOUSE_CELLS, "human_cells": repo / HUMAN_CELLS},
    outputs=[paths["png"], paths["png_600dpi"], paths["svg"]],
    font=font,
)
record["panel"] = style.panel_spec(LETTER)
record["nuclei_drawn"] = drawn_counts
style.write_json(paths["provenance"], record)
LOG.info("Wrote %s", paths["provenance"])